In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# شغّل أول دائرة لك على العتاد الفعلي

<Accordion>
<AccordionItem title="Package versions">

الكود الموجود في هذه الصفحة تم تطويره باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
يحتوي هذا المثال على قسمين. ستُنشئ أولاً برنامجاً كمومياً بسيطاً يُشبه "Hello world" وتُشغّله على وحدة المعالجة الكمومية (QPU). ولأن البحث الكمومي الفعلي يتطلب برامج أكثر قوة ومتانة، ستقوم في القسم الثاني ([التوسع لأعداد كبيرة من qubits](#scale-to-large-numbers-of-qubits)) بتطوير البرنامج البسيط ليصل إلى مستوى الاستخدام الفعلي.

## التثبيت والمصادقة
1. إذا لم تكن قد ثبّتَ Qiskit بعد، فستجد التعليمات في دليل [البداية السريعة](/guides/quick-start).

    - ثبّت Qiskit Runtime لتشغيل المهام على أجهزة الكم:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - أعدّ بيئة لتشغيل دفاتر Jupyter محلياً:

        ```bash
        pip install jupyter
        ```

2. أعدّ مصادقتك للوصول إلى أجهزة الكم من خلال [الخطة المجانية المفتوحة](/guides/plans-overview#open-plan).

    (إذا تلقيتَ دعوة بالبريد الإلكتروني للانضمام إلى حساب، اتبع [خطوات المستخدمين المدعوّين](/guides/cloud-setup-invited) عوضاً عن ذلك.)

    - اذهب إلى [منصة IBM Quantum](https://quantum.cloud.ibm.com/) لتسجيل الدخول أو إنشاء حساب.
         > **Note:** إذا كنت تتصل عبر خادم وكيل (proxy)، يجب عليك استخدام Qiskit Runtime الإصدار v0.44.0 أو أحدث.
    - أنشئ مفتاح API الخاص بك (يُعرف أيضاً بـ *رمز API*) من [لوحة التحكم](https://quantum.cloud.ibm.com/)، ثم انسخه إلى مكان آمن.
    - اذهب إلى صفحة [Instances](https://quantum.cloud.ibm.com/instances) وابحث عن النسخة التي تريد استخدامها. مرّر الماوس فوق CRN الخاص بها وانقر لنسخه.

    - احفظ بياناتك للمصادقة محلياً باستخدام هذا الكود:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
            # For `token`, use the 44-character API_KEY you created
            # and saved from the IBM Quantum Platform Home dashboard
            token="<your-api-key>",
            instance="<CRN>", # Optional
        )
        ```

3. الآن يمكنك استخدام كود Python هذا في أي وقت تريد فيه المصادقة على خدمة Qiskit Runtime:
    ```python
    from qiskit_ibm_runtime import QiskitRuntimeService

    # Run every time you need the service
    service = QiskitRuntimeService()
    ```
> **Info:** إذا كنت تستخدم حاسوباً عاماً أو بيئة غير آمنة، اتبع [تعليمات المصادقة اليدوية](/guides/cloud-setup-untrusted) عوضاً عن ذلك للحفاظ على أمان بيانات مصادقتك.

## إنشاء وتشغيل برنامج كمومي بسيط
الخطوات الأربع لكتابة برنامج كمومي باستخدام أنماط Qiskit هي:

1.  تحويل المسألة إلى صيغة كمومية أصيلة.

2.  تحسين الدوائر والمؤثِّرات.

3.  التنفيذ باستخدام دالة كمومية أساسية (primitive).

4.  تحليل النتائج.

### الخطوة 1. تحويل المسألة إلى صيغة كمومية أصيلة
في البرنامج الكمومي، تُمثّل *الدوائر الكمومية (quantum circuits)* الصيغة الأصيلة لتمثيل التعليمات الكمومية، بينما تُمثّل *المؤثِّرات (operators)* الكميات القابلة للقياس. عند إنشاء دائرة، عادةً ما تُنشئ كائن [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) جديداً، ثم تُضيف إليه التعليمات بالتسلسل.
ينشئ الكود التالي دائرة تُولّد *حالة Bell*، وهي حالة تكون فيها قبّيتان (Qubits) في تشابك تام مع بعضهما البعض.

> **Note:** يستخدم Qiskit SDK ترقيم البتات LSb 0 حيث تحمل الخانة $n^{th}$ القيمة $1 \ll n$ أو $2^n$. لمزيد من التفاصيل، راجع موضوع [ترتيب البتات في Qiskit SDK](/guides/bit-ordering).

In [1]:
# This cell is hidden from users.  It hides several unnecessary warnings.

import warnings
import logging

warnings.filterwarnings("ignore", message=".*Instance was not set*")
warnings.filterwarnings("ignore", message=".*loading instance*")
warnings.filterwarnings("ignore", message=".*using instance*")

# This cell is hidden from users.  It hides several unnecessary warnings.


class IgnoreSpecificMessages(logging.Filter):
    def filter(self, record):
        for text in [
            "Instance was not set",
            "Loading default saved account",
            "Loading instance",
            "Instance was not set",
            "using instance",
        ]:
            if text in record.getMessage():
                return False
        return True


logger = logging.getLogger("qiskit_ibm_runtime")
logger.addFilter(IgnoreSpecificMessages())

for handler in logger.handlers:
    handler.addFilter(IgnoreSpecificMessages())

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

> **Note:** هنا، المؤثِّر مثل `ZZ` هو اختصار للجداء التنسوري $Z\otimes Z$، مما يعني قياس Z على Qubit 1 وZ على Qubit 0 معاً، والحصول على معلومات حول الترابط بين Qubit 1 وQubit 0. قيم التوقع مثل هذه تُكتب أيضاً عادةً كـ $\langle Z_1 Z_0 \rangle$.
> 
> إذا كانت الحالة متشابكة، فينبغي أن يختلف قياس $\langle Z_1 Z_0 \rangle$ عن قياس $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. بالنسبة للحالة المتشابكة المحددة التي تُنشئها دائرتنا الموصوفة أعلاه، يجب أن يكون قياس $\langle Z_1 Z_0 \rangle$ مساوياً لـ 1 وقياس $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ مساوياً للصفر.
<span id="optimize"></span>

### الخطوة 2. تحسين الدوائر والمؤثِّرات

عند تنفيذ الدوائر على جهاز، من المهم تحسين مجموعة التعليمات التي تحتوي عليها الدائرة وتقليل العمق الكلي (عدد التعليمات تقريباً) للدائرة. يضمن ذلك الحصول على أفضل النتائج الممكنة من خلال تقليل تأثيرات الخطأ والضوضاء. علاوة على ذلك، يجب أن تتوافق تعليمات الدائرة مع [مجموعة تعليمات معمارية الجهاز (ISA)](/guides/transpile#instruction-set-architecture) الخاص بالـ Backend، مع مراعاة البوابات الأساسية وتوصيلية Qubits في الجهاز.

يُهيئ الكود التالي جهازاً حقيقياً لإرسال مهمة إليه ويحوّل الدائرة والمؤثِّرات لتتوافق مع ISA ذلك الـ Backend. يتطلب ذلك أن تكون قد [حفظت بياناتك للمصادقة](/guides/cloud-setup) مسبقاً.

In [3]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### الخطوة 3. التنفيذ باستخدام الدوال الكمومية الأساسية

يمكن للحواسيب الكمومية أن تُنتج نتائج عشوائية، لذا عادةً ما تجمع عيّنة من المخرجات بتشغيل الدائرة عدة مرات. يمكنك تقدير قيمة المؤثِّر باستخدام فئة `Estimator`. وتُعدّ `Estimator` واحدة من اثنتين من [الدوال الأساسية (primitives)](/guides/get-started-with-estimator)؛ الأخرى هي `Sampler`، التي يمكن استخدامها للحصول على بيانات من حاسوب كمومي. تمتلك هذه الكائنات طريقة `run()` التي تُنفّذ مجموعة الدوائر والمؤثِّرات والمعاملات (إن وُجدت)، باستخدام [كتلة موحّدة أساسية (PUB)](/guides/get-started-with-sampler).

In [4]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

### Step 3. Execute using the quantum primitives

Quantum computers can produce random results, so you usually collect a sample of the outputs by running the circuit many times. You can estimate the value of the observable by using the `Estimator` class. `Estimator` is one of two [primitives](/docs/guides/get-started-with-estimator); the other is `Sampler`, which can be used to get data from a quantum computer.  These objects possess a `run()` method that executes the selection of circuits, observables, and parameters (if applicable), using a [primitive unified bloc (PUB)](/docs/guides/get-started-with-sampler).

In [5]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d8286mfoha1c73bl8hrg


بعد إرسال المهمة، يمكنك الانتظار حتى تكتمل المهمة ضمن نسخة Python الحالية، أو استخدام `job_id` لاسترجاع البيانات لاحقاً. (راجع [قسم استرجاع المهام](/guides/monitor-job#retrieve-job-results-at-a-later-time) للتفاصيل.)

بعد اكتمال المهمة، افحص مخرجاتها من خلال الخاصية `result()` للمهمة.

In [6]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [7]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

:::note[Alternative: run the example using a simulator]

  عند تشغيل برنامجك الكمومي على جهاز حقيقي، يجب على حِملك الانتظار في قائمة الانتظار قبل التشغيل. لتوفير الوقت، يمكنك بدلاً من ذلك استخدام الكود التالي لتشغيل هذا الحِمل الصغير على [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) مع وضع الاختبار المحلي في Qiskit Runtime. لاحظ أن هذا ممكن فقط للدوائر الصغيرة. عند التوسع في القسم التالي، ستحتاج إلى استخدام جهاز حقيقي.

In [8]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

:::
### الخطوة 4. تحليل النتائج
خطوة التحليل هي عادةً المرحلة التي قد تُعالج فيها نتائجك باستخدام، على سبيل المثال، تخفيف أخطاء القياس أو استقراء الضوضاء الصفرية (ZNE). قد تُغذّي هذه النتائج في سير عمل آخر لمزيد من التحليل أو تُعدّ رسماً بيانياً للقيم والبيانات الرئيسية. بشكل عام، هذه الخطوة خاصة بمسألتك. في هذا المثال، ارسم كل قيم التوقع التي تم قياسها لدائرتنا.

يمكن الوصول إلى قيم التوقع والانحرافات المعيارية للمؤثِّرات التي حددتها لـ Estimator من خلال الخاصيتين `PubResult.data.evs` و`PubResult.data.stds` لنتيجة المهمة. للحصول على نتائج Sampler، استخدم الدالة `PubResult.data.meas.get_counts()`، التي ستُرجع `dict` من القياسات على شكل سلاسل بتية كمفاتيح وعدد مرات ظهورها كقيم مقابلة. لمزيد من المعلومات، راجع دليل [البدء السريع مع Sampler](/guides/get-started-with-sampler).

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
  observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission. You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

لاحظ أنه بالنسبة لـ Qubits 0 و1، فإن قيم التوقع المستقلة لكل من X وZ تساوي 0، بينما الترابطات (`XX` و`ZZ`) تساوي 1. هذه علامة مميزة للتشابك الكمومي.

In [9]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

## التوسع إلى أعداد كبيرة من Qubits
في الحوسبة الكمية، يُعدّ العمل على المستوى الاستخداميّ (utility-scale) خطوةً أساسيةً للتقدم في هذا المجال. يستلزم هذا النوع من العمل إجراء حسابات على نطاق أوسع بكثير؛ إذ قد تشمل الدوائر ما يزيد على 100 Qubit وأكثر من 1000 بوابة. يوضّح هذا المثال كيف يمكنك إنجاز عمل على المستوى الاستخداميّ باستخدام معالجات IBM&reg; الكمية (QPUs)، وذلك بإنشاء وتحليل حالة GHZ مكوّنة من 100 Qubit. يعتمد المثال على سير عمل أنماط Qiskit، وينتهي بقياس قيمة التوقع $\langle Z_0 Z_i \rangle$ لكل Qubit.

### الخطوة 1. تعريف المسألة
اكتب دالةً تُعيد `QuantumCircuit` يُجهّز حالة GHZ مؤلّفة من $n$ Qubit (وهي في جوهرها امتداد لحالة Bell)، ثم استخدم تلك الدالة لإعداد حالة GHZ المؤلّفة من 100 Qubit، واجمع المراقبات (observables) التي ستقيسها.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc)
        for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state,
            assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with 100 qubits in the GHZ state
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

بعد ذلك، عيّن عوامل التشغيل (operators) التي تهمّك. يستخدم هذا المثال عوامل `ZZ` بين الـ Qubits لدراسة السلوك مع زيادة المسافة بينها. ستكشف قيم التوقع المتدهورة بين الـ Qubits البعيدة عن مستوى الضوضاء الموجود.

In [11]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]

operators = [SparsePauliOp(operator) for operator in operator_strings]

### Step 2. Optimize the problem for execution on quantum hardware

The following code transforms the circuit and observables to match the backend's ISA. It requires that you have already [saved your credentials](/docs/guides/cloud-setup)

In [12]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### الخطوة 2. تحسين المسألة للتنفيذ على العتاد الكمي
يحوّل الكود التالي الدائرة والمراقبات لتتناسب مع ISA الخاصة بالـ Backend. يشترط ذلك أن تكون قد [حفظت بيانات اعتمادك](/guides/cloud-setup) مسبقاً.

In [13]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [14]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d828aeo0bvlc73d1vs20


### Step 4. Post-process results

After the job completes, plot the results and notice that $\langle Z_0 Z_i \rangle$ decreases with increasing $i$, even though in an ideal simulation all $\langle Z_0 Z_i \rangle$ should be 1.

In [15]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment](/docs/guides/online-lab-environments).
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>